# Лабораторна 6. RAG-довідка для CLI «Картки з конспекту»

Користувачі застосунку з лабораторної 4 можуть запитати про запуск, параметри, формат, помилки й результати перевірок. Відповідь містить посилання на документ і цитату. У Colab виберіть T4 GPU та виконайте клітинки зверху вниз.

Корпус складається із шести коротких оригінальних україномовних Markdown-конспектів. У вихідному репозиторії окремої ліцензії не вказано. Нижче наведено власні резюме поведінки програми з посиланнями на файли зафіксованої версії. Версія матеріалів — d0a317fb2a66eae571c6fc1f1b755e053711bf23.

Коли зміняться параметри CLI, достатньо оновити конспекти й переіндексувати їх. Промпт з усіма документами можна використати на маленькому корпусі, але він передавав би весь текст для кожного запиту. RAG повертає короткий контекст і дає змогу виміряти, чи знайдений конкретний доказ. Донавчання не прив'язує твердження до версії документа.

In [1]:
from dataclasses import dataclass

REVISION = "d0a317fb2a66eae571c6fc1f1b755e053711bf23"
REPO = f"https://github.com/Xlntaro/ai_systems/blob/{REVISION}/lab4/app"

@dataclass(frozen=True)
class Document:
    id: str
    title: str
    path: str
    sections: tuple

corpus = [
    Document("setup", "Запуск і підключення до Ollama", "README.md#запуск-на-nixos", (
        ("terminals", "Спочатку в першому терміналі запустіть сервер командою lab4-serve. У другому терміналі відкрийте dev shell і виконайте ollama pull для завантаження моделі. Далі запускайте скрипт lab4/app/main.py."),
        ("requirements", "За межами Nix необхідні Python 3.10 або новіший та локальний Ollama. Для клієнтської частини вистачає стандартної бібліотеки Python."),
        ("network", "CLI надсилає запити на http://127.0.0.1:11434/v1/chat/completions. Для Ollama поза Nix потрібен OLLAMA_CONTEXT_LENGTH=4096."),
    )),
    Document("invocation", "Формат вводу та режими виводу", "README.md#картки-з-конспекту", (
        ("notes", "CLI приймає нотатки українською або англійською. Передайте шлях до файла UTF-8 як позиційний аргумент. Для stdin використовуйте аргумент «-» або пропустіть позиційний аргумент."),
        ("output", "CLI за замовчуванням друкує нумеровані запитання і відповіді українською. Опція --json повертає машинозчитуваний JSON-масив карток."),
        ("override", "Опція --model дає змогу вибрати локальний тег моделі для одного запуску. Встановіть обрану модель в Ollama перед виконанням."),
    )),
    Document("generation", "Ліміти та відтворюваність генерації", "README.md#параметри", (
        ("sampling", "Параметр temperature=0.1 зменшує варіативність генерації. Початковий стан семплінгу задає seed=1."),
        ("limits", "Вхід із нотатками обмежено 3000 байтами UTF-8. Серверний контекст Ollama становить 4096 токенів, а відповідь має бюджет max_tokens=384."),
        ("timing", "Таймаут запиту до Ollama — 60 секунд. Під час очікування CLI показує «Генерую картки…» у stderr."),
    )),
    Document("validation", "Перевірка й обробка карток", "main.py#L77", (
        ("schema", "Клієнт читає JSON з ключем cards. Значення має бути масивом максимум із чотирьох об'єктів. Кожен об'єкт має містити непорожні текстові question та answer."),
        ("cloze", "Якщо модель повернула твердження, дослівно присутнє в конспекті, CLI може зробити картку з пропуском. У пропуск потрапляють два останні слова твердження. Інші твердження в полі question відкидаються."),
        ("truncation", "Якщо finish_reason дорівнює length, CLI повідомляє, що відповідь обірвалася на ліміті, і радить скоротити конспект."),
    )),
    Document("errors", "Діагностика помилок CLI", "main.py#L58", (
        ("unavailable", "HTTP 404 від Ollama означає, що вказана модель недоступна. Для відновлення треба завантажити відповідний тег командою ollama pull."),
        ("connection", "Якщо Ollama не відповідає через недоступне з'єднання, CLI радить запустити lab4-serve і повторити запит. Якщо вичерпано час очікування, Ollama не відповів вчасно."),
        ("empty", "Порожній або складений лише з пробілів ввід завершується повідомленням «Конспект порожній». Файл з некоректним UTF-8 призводить до помилки читання."),
    )),
    Document("quality", "Моделі та перевірка якості карток", "README.md#матеріали-перевірки", (
        ("models", "Модель за замовчуванням — qwen2.5:1.5b-instruct-q4_K_M. У лабораторній її зіставили з більшим тегом qwen2.5:3b-instruct-q4_K_M."),
        ("scores", "За повним виконанням критеріїв модель 1.5B отримала 7/10, а модель 3B — 2/10. Порівняння зроблено на десяти прикладах з лабораторної 4."),
        ("records", "Промпти та критерії записані у quality_cases.json. Фактичні відповіді та оцінки обох моделей містить quality_results.json."),
    )),
]
bytes_count = sum(len(("# " + d.title + "\n\n" + "\n\n".join(t for _, t in d.sections)).encode()) for d in corpus)
print(f"{len(corpus)} Markdown-документів, {sum(len(d.sections) for d in corpus)} розділів, {bytes_count} байтів UTF-8.")

6 Markdown-документів, 18 розділів, 4540 байтів UTF-8.


## 1.2. Бенчмарк

Фіксую 26 запитань до налаштування індексу. Серед них є точні команди, числа, перефразування, багатоскладові запитання й шість запитань поза корпусом. Для кожного питання з відповіддю є еталон, розділ та доказ. Запити без доказу перевіряють відмову окремо від ретриву.

In [2]:
@dataclass(frozen=True)
class Case:
    question: str
    section: str | None = None
    reference: str | None = None
    evidence: str | None = None
    required: tuple = ()

benchmark = [
    Case("Яку команду ввести, щоб підняти сервер перш ніж запускати клієнт?", "setup/terminals", "lab4-serve", "lab4-serve", (r"lab4-serve",)),
    Case("Які вимоги для запуску без Nix?", "setup/requirements", "Python 3.10+ та локальний Ollama.", "Python 3.10 або новіший", (r"3\.10", r"ollama")),
    Case("На яку повну HTTP-адресу CLI надсилає запити?", "setup/network", "http://127.0.0.1:11434/v1/chat/completions", "http://127.0.0.1:11434/v1/chat/completions", (r"127\.0\.0\.1:11434/v1/chat/completions",)),
    Case("Як подати конспект зі стандартного вводу?", "invocation/notes", "Передати '-' або пропустити позиційний аргумент.", "Для stdin використовуйте аргумент", (r"[-−]|пропуст|без аргумент",)),
    Case("Як отримати результат, придатний для автоматичного читання іншою програмою?", "invocation/output", "Використати --json, щоб отримати масив карток.", "Опція --json", (r"--json",)),
    Case("Як тимчасово вибрати іншу локальну модель під час запуску?", "invocation/override", "Передати --model з локальним тегом.", "Опція --model", (r"--model",)),
    Case("Які температура та seed задані для генерації карток?", "generation/sampling", "temperature=0.1 та seed=1.", "temperature=0.1", (r"0[.,]1", r"\b1\b")),
    Case("Як обмежено байтову довжину нотаток?", "generation/limits", "3000 байтів UTF-8.", "3000 байтами UTF-8", (r"3\s?000", r"байт")),
    Case("Які розміри контексту сервера і бюджету на відповідь?", "generation/limits", "4096 токенів контексту, 384 токени на відповідь.", "max_tokens=384", (r"4\s?096", r"\b384\b")),
    Case("Скільки клієнт чекає завершення запиту до Ollama?", "generation/timing", "60 секунд.", "60 секунд", (r"\b60\b",)),
    Case("Які ключі має об'єкт картки у відповіді?", "validation/schema", "Непорожні question і answer.", "question та answer", (r"question", r"answer")),
    Case("Скільки найбільше карток може прийняти клієнт?", "validation/schema", "Чотири.", "максимум із чотирьох", (r"\b4\b|чотир",)),
    Case("Які слова вилучає CLI, коли перетворює твердження на картку з пропуском?", "validation/cloze", "Два останні слова.", "два останні слова", (r"два|\b2\b", r"остан")),
    Case("Що означає finish_reason=length і що радить CLI?", "validation/truncation", "Відповідь досягла ліміту, треба скоротити конспект.", "скоротити конспект", (r"ліміт|обірвал", r"скорот")),
    Case("Що зробити, коли сервер повертає HTTP 404 для моделі?", "errors/unavailable", "Завантажити потрібний тег через ollama pull.", "командою ollama pull", (r"ollama pull",)),
    Case("Яке повідомлення з'явиться, якщо надіслати лише пробіли?", "errors/empty", "Конспект порожній.", "Конспект порожній", (r"конспект порожній",)),
    Case("Який локальний тег використовує програма за замовчуванням?", "quality/models", "qwen2.5:1.5b-instruct-q4_K_M.", "qwen2.5:1.5b-instruct-q4_K_M", (r"qwen2\.5:1\.5b-instruct-q4_k_m",)),
    Case("Який повний результат у тесті якості здобули моделі 1.5B і 3B?", "quality/scores", "1.5B — 7/10, 3B — 2/10.", "7/10, а модель 3B — 2/10", (r"7\s*(?:/|з|із)\s*10", r"2\s*(?:/|з|із)\s*10")),
    Case("У якому файлі шукати фактичні відповіді й оцінки?", "quality/records", "quality_results.json.", "quality_results.json", (r"quality_results\.json",)),
    Case("Що запустити, якщо CLI не може встановити з'єднання з Ollama?", "errors/connection", "Запустити lab4-serve й повторити запит.", "запустити lab4-serve", (r"lab4-serve",)),
    Case("Який тариф за тисячу токенів у хмарній версії CLI?"),
    Case("Як експортувати картки у форматі Anki APKG?"),
    Case("Які облікові дані потрібні для вебкабінету?"),
    Case("Чи можна подати скан PDF і який OCR використовується?"),
    Case("Яка точність генерації на довгих конспектах німецькою мовою?"),
    Case("Яку команду виконати, щоб опублікувати картки для одногрупників?"),
]
sections = {f"{d.id}/{section}": text for d in corpus for section, text in d.sections}
assert len(benchmark) == len({case.question for case in benchmark}) == 26
assert all(case.evidence in sections[case.section] for case in benchmark if case.section)
print(f"{len(benchmark)} питань, із них {sum(case.section is None for case in benchmark)} поза корпусом.")

26 питань, із них 6 поза корпусом.


## 1.3–1.4. Індекс та конфігурації пошуку

Для української мови використовую intfloat/multilingual-e5-small з префіксами query: і passage:. L2-нормалізація дає змогу обчислити косинус через скалярний добуток. Матриця NumPy в оперативній пам'яті дає точний пошук для невеликого корпусу. Вікно до 64 слів з перекриттям 12 слів застосовую в межах одного тематичного розділу. Заголовок, ідентифікатор розділу і URL лишаються у метаданих.

Порівнюю векторний пошук із гібридним пошуком, який додає BM25 через RRF. BM25 може підвищити ранг фрагментів з точними командами, параметрами й іменами файлів. Для кожного методу вимірюю hit@1, hit@3, hit@5 і MRR до глибини 5.

In [3]:
%pip -q install "sentence-transformers>=5,<6" "transformers>=4.57,<5" rank-bm25 ipywidgets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
from collections import defaultdict
import os
import re

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from IPython.display import Markdown, display

if not torch.cuda.is_available():
    torch.set_num_threads(min(torch.get_num_threads(), 4))

@dataclass(frozen=True)
class Chunk:
    id: str
    section: str
    title: str
    text: str
    url: str

CHUNK_WORDS, OVERLAP = 64, 12
chunks = []
for doc in corpus:
    for section, text in doc.sections:
        words = text.split()
        for start in range(0, len(words), CHUNK_WORDS - OVERLAP):
            end = min(start + CHUNK_WORDS, len(words))
            ident = f"{doc.id}/{section}"
            chunks.append(Chunk(f"{ident}/{start}", ident, doc.title,
                                " ".join(words[start:end]), f"{REPO}/{doc.path}"))
            if end == len(words):
                break

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=device)
texts = [f"{chunk.title}. {chunk.text}" for chunk in chunks]
vectors = embedder.encode([f"passage: {text}" for text in texts], normalize_embeddings=True, show_progress_bar=False)

def tokens(text):
    return re.findall(r"\w+(?:[.'’-]\w+)*", text.casefold())

bm25 = BM25Okapi([tokens(text) for text in texts])

def order(scores):
    return np.argsort(-np.asarray(scores), kind="stable").tolist()

def retrieve(question, method="hybrid", k=5):
    query = embedder.encode(f"query: {question}", normalize_embeddings=True, show_progress_bar=False)
    semantic = order(vectors @ query)
    if method == "dense":
        return [chunks[i] for i in semantic[:k]]
    if method != "hybrid":
        raise ValueError(f"Невідомий метод: {method}")
    lexical_scores = bm25.get_scores(tokens(question))
    lexical = [i for i in order(lexical_scores) if lexical_scores[i] > 0]
    scores = defaultdict(float)
    for ranking in (semantic[:10], lexical[:10]):
        for rank, i in enumerate(ranking, 1):
            scores[i] += 1 / (60 + rank)
    return [chunks[i] for i in sorted(scores, key=scores.get, reverse=True)[:k]]

def valid_evidence(chunk, case):
    return chunk.section == case.section and case.evidence is not None and case.evidence.casefold() in chunk.text.casefold()

def rank_of_evidence(retrieved, case):
    return next((rank for rank, chunk in enumerate(retrieved, 1) if valid_evidence(chunk, case)), None)

answerable = [case for case in benchmark if case.section]
retrieval_rows = []
for method in ("dense", "hybrid"):
    ranks = [rank_of_evidence(retrieve(case.question, method), case) for case in answerable]
    retrieval_rows.append({
        "метод": method,
        **{f"hit@{k}": sum(rank is not None and rank <= k for rank in ranks) / len(ranks) for k in (1, 3, 5)},
        "MRR@5": sum(1 / rank for rank in ranks if rank) / len(ranks),
    })
retrieval_metrics = pd.DataFrame(retrieval_rows).set_index("метод")
display(retrieval_metrics.round(3))
METHOD = max(("dense", "hybrid"), key=lambda method: retrieval_metrics.loc[method, "MRR@5"])
print(f"Індекс: {len(chunks)} фрагментів, форма {vectors.shape}, пристрій {device}.")
print(f"Максимальна фактична довжина {max(len(c.text.split()) for c in chunks)} слів. Метод за MRR@5: {METHOD}.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

,hit@1,hit@3,hit@5,MRR@5
метод,,,,
dense,0.75,0.85,0.85,0.792
hybrid,0.85,0.90,0.95,0.888


Індекс: 18 фрагментів, форма (18, 384), пристрій cuda.
Максимальна фактична довжина 27 слів. Метод за MRR@5: hybrid.


## 1.5. Генерація, перевірка цитат та оцінка

Модель отримує п'ять фрагментів і повертає відповідь, ідентифікатор та дослівну цитату. Клієнт перевіряє ідентифікатор, дослівність цитати й наявність чисел із відповіді у цитаті. Якщо структура або цитата не пройшли перевірку, застосунок повідомляє, що відповіді немає. Збіг цитати підтверджує походження тексту. Семантичну правильність оцінюю окремо.

Клієнт використовує OpenAI-сумісний маршрут /v1/chat/completions. Для Ollama з лабораторної 4 задайте RAG_API_BASE=http://127.0.0.1:11434/v1 і RAG_MODEL=qwen2.5:1.5b-instruct-q4_K_M. У Colab за відсутності зовнішнього API запускається HTTP-сервер на loopback із моделлю Qwen/Qwen3-4B-Instruct-2507 через Transformers. Зовнішній API за потреби читає ключ зі змінної RAG_API_KEY.

In [13]:
import gc
import json
from http.server import BaseHTTPRequestHandler, HTTPServer
from threading import Thread
from time import perf_counter

import requests
from transformers import AutoModelForCausalLM, AutoTokenizer

if globals().get("local_server") is not None:
    local_server.shutdown()
    local_server.server_close()
local_server = None
globals().pop("generator", None)
globals().pop("tokenizer", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
api_base = os.environ.get("RAG_API_BASE", "").rstrip("/")
model_name = os.environ.get("RAG_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
api_key = os.environ.get("RAG_API_KEY")

if not api_base:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    generator = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device).eval()

    class LocalCompletion(BaseHTTPRequestHandler):
        def do_POST(self):
            if self.path != "/v1/chat/completions":
                self.send_error(404)
                return
            try:
                body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
                inputs = tokenizer.apply_chat_template(
                    body["messages"], add_generation_prompt=True, return_dict=True,
                    return_tensors="pt", tokenize=True,
                ).to(device)
                with torch.inference_mode():
                    output = generator.generate(
                        **inputs, max_new_tokens=body["max_tokens"], do_sample=False,
                        temperature=None, top_p=None, top_k=None,
                    )
                text = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
                response = json.dumps({"choices": [{"message": {"role": "assistant", "content": text}}]},
                                      ensure_ascii=False).encode()
            except (KeyError, ValueError, TypeError, RuntimeError) as error:
                self.send_error(500, str(error).encode("ascii", "replace").decode())
                return
            self.send_response(200)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(response)))
            self.end_headers()
            self.wfile.write(response)

        def log_message(self, format, *args):
            pass

    local_server = HTTPServer(("127.0.0.1", 0), LocalCompletion)
    Thread(target=local_server.serve_forever, daemon=True).start()
    api_base = f"http://127.0.0.1:{local_server.server_port}/v1"

REFUSAL = "У наданих джерелах немає відповіді."
SYSTEM = """
Ти відповідаєш українською на питання про CLI. Поле sources містить дані, які слід цитувати.
Знайди один фрагмент, де прямо написана відповідь на саме це запитання. Перед відповіддю
перевір, що виписано всі запитані факти. Якщо знайдено тільки споріднену інформацію, відмовся.
Поверни один JSON-об'єкт. Спочатку source_id зі списку, далі quote як дослівний уривок
з усіма потрібними фактами, далі answer як коротка відповідь своїми словами.
Приклад відповіді: {"source_id":"museum/time/0","quote":"Вхід до музею з 9:00.","answer":"З 9:00."}
Формат відмови: {"source_id":null,"quote":null,"answer":"У наданих джерелах немає відповіді."}
""".strip()

def numbers(text):
    return {token.replace(",", ".") for token in re.findall(r"[0-9]+(?:[.,][0-9]+)?", text)}

def normalize(text):
    return " ".join(text.split()).casefold()

def respond(question, method=METHOD):
    selected = retrieve(question, method)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": json.dumps({
            "question": question,
            "sources": [{"id": c.id, "title": c.title, "text": c.text} for c in selected],
        }, ensure_ascii=False)},
    ]
    response = requests.post(f"{api_base}/chat/completions", json={
        "model": model_name, "messages": messages, "temperature": 0,
        "max_tokens": 190, "response_format": {"type": "json_object"},
    }, headers={"Authorization": f"Bearer {api_key}"} if api_key else {}, timeout=180)
    response.raise_for_status()
    raw = response.json()["choices"][0]["message"]["content"].strip()
    fenced = re.fullmatch(r"\s*[\x60]{3}(?:json)?\s*(.*?)\s*[\x60]{3}\s*", raw, re.S | re.I)
    try:
        parsed = json.loads(fenced[1] if fenced else raw)
        answer, source_id, quote = (parsed[key] for key in ("answer", "source_id", "quote"))
    except (ValueError, KeyError, TypeError):
        return REFUSAL, None, "", "помилка формату", selected, raw
    if isinstance(answer, str) and normalize(answer) == normalize(REFUSAL) and source_id is None:
        return REFUSAL, None, "", "відмова моделі", selected, raw
    source = next((c for c in selected if c.id == source_id), None)
    if (source and isinstance(answer, str) and answer.strip()
            and isinstance(quote, str) and len(normalize(quote)) >= 10
            and normalize(quote) in normalize(source.text)
            and numbers(answer) <= numbers(quote)):
        return answer.strip(), source, quote.strip(), "відповідь", selected, raw
    return REFUSAL, None, "", "відхилено цитату або джерело", selected, raw

print(f"Генератор: {model_name}. Пристрій для локального інференсу: {device}.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Генератор: Qwen/Qwen3-4B-Instruct-2507. Пристрій для локального інференсу: cuda.


In [14]:
rows = []
started = perf_counter()
for number, case in enumerate(benchmark, 1):
    answer, source, quote, status, found, raw = respond(case.question)
    refused = source is None
    answer_ok = refused if case.section is None else (
        not refused and all(re.search(pattern, answer, re.I) for pattern in case.required)
    )
    citation_ok = None if case.section is None else bool(
        source and valid_evidence(source, case) and normalize(case.evidence) in normalize(quote)
    )
    rows.append({
        "№": number, "питання": case.question, "еталон": case.reference,
        "відповідь": answer, "цитата": quote, "джерело": source.id if source else None,
        "посилання": source.url if source else None, "ранг доказу": rank_of_evidence(found, case),
        "стан": status, "збіг ознак": answer_ok, "цитування": citation_ok,
        "сирий вивід": raw,
    })
    print(f"{number:02} | {status} | {source.id if source else '-'} | {answer}", flush=True)

results = pd.DataFrame(rows)
measured = results[results["еталон"].notna()]
outside = results[results["еталон"].isna()]
print(f"\nГенерація бенчмарку: {perf_counter() - started:.1f} с")
print(f"Відповіді з потрібними ознаками: {int(measured['збіг ознак'].sum())}/{len(measured)}")
print(f"Цитати з потрібним доказом: {int(measured['цитування'].sum())}/{len(measured)}")
print(f"Правильні відмови: {int(outside['збіг ознак'].sum())}/{len(outside)}")
display(results.drop(columns="сирий вивід"))

01 | відповідь | setup/terminals/0 | Команду lab4-serve ввести, щоб підняти сервер перш ніж запускати клієнт.
02 | відповідь | setup/requirements/0 | Для запуску за межами Nix необхідні Python 3.10 або новіший та локальний Ollama.
03 | відповідь | setup/network/0 | CLI надсилає запити на повну HTTP-адресу http://127.0.0.1:11434/v1/chat/completions.
04 | відповідь | invocation/notes/0 | Щоб подати конспект зі стандартного вводу, використовуйте аргумент «-» або пропустіть позиційний аргумент.
05 | відмова моделі | - | У наданих джерелах немає відповіді.
06 | відповідь | invocation/override/0 | Щоб тимчасово вибрати іншу локальну модель під час запуску, використовуйте опцію --model для вибору тегу моделі, а потім встановіть цю модель в Ollama перед виконанням запиту.
07 | відповідь | generation/sampling/0 | Температура задана як 0.1, а seed — 1.
08 | відповідь | generation/limits/0 | Байтову довжину нотаток обмежено 3000 байтами UTF-8.
09 | відповідь | generation/limits/0 | Розмір контекс

,№,питання,еталон,відповідь,цитата,джерело,посилання,ранг доказу,стан,збіг ознак,цитування
0,1,"Яку команду ввести, щоб підняти сервер перш ні...",lab4-serve,"Команду lab4-serve ввести, щоб підняти сервер ...",Спочатку в першому терміналі запустіть сервер ...,setup/terminals/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
1,2,Які вимоги для запуску без Nix?,Python 3.10+ та локальний Ollama.,Для запуску за межами Nix необхідні Python 3.1...,За межами Nix необхідні Python 3.10 або новіши...,setup/requirements/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
2,3,На яку повну HTTP-адресу CLI надсилає запити?,http://127.0.0.1:11434/v1/chat/completions,CLI надсилає запити на повну HTTP-адресу http:...,CLI надсилає запити на http://127.0.0.1:11434/...,setup/network/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
3,4,Як подати конспект зі стандартного вводу?,Передати '-' або пропустити позиційний аргумент.,"Щоб подати конспект зі стандартного вводу, вик...",CLI приймає нотатки українською або англійсько...,invocation/notes/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
4,5,"Як отримати результат, придатний для автоматич...","Використати --json, щоб отримати масив карток.",У наданих джерелах немає відповіді.,,None,None,NaN,відмова моделі,False,False
5,6,Як тимчасово вибрати іншу локальну модель під ...,Передати --model з локальним тегом.,Щоб тимчасово вибрати іншу локальну модель під...,Опція --model дає змогу вибрати локальний тег ...,invocation/override/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
6,7,Які температура та seed задані для генерації к...,temperature=0.1 та seed=1.,"Температура задана як 0.1, а seed — 1.",Параметр temperature=0.1 зменшує варіативність...,generation/sampling/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
7,8,Як обмежено байтову довжину нотаток?,3000 байтів UTF-8.,Байтову довжину нотаток обмежено 3000 байтами ...,Вхід із нотатками обмежено 3000 байтами UTF-8.,generation/limits/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
8,9,Які розміри контексту сервера і бюджету на від...,"4096 токенів контексту, 384 токени на відповідь.","Розмір контексту сервера — 4096 токенів, бюдже...",Вхід із нотатками обмежено 3000 байтами UTF-8....,generation/limits/0,https://github.com/Xlntaro/ai_systems/blob/d0a...,1.0,відповідь,True,True
9,10,Скільки клієнт чекає завершення запиту до Ollama?,60 секунд.,У наданих джерелах немає відповіді.,,None,None,2.0,відмова моделі,False,False


## 1.6. Аналіз відмов

Розрізняю відсутність еталонного доказу в топ-5, неправильну генерацію за наявності доказу, непідтверджену цитату, помилку формату і питання поза корпусом. Оцінювання ключових ознак можна відтворити, але воно не виявляє всіх неправильних тверджень. Випадки з кількома фактами потребують окремої перевірки всього тексту.

In [15]:
def diagnose(row):
    if pd.isna(row["еталон"]):
        if row["збіг ознак"]:
            return "поза корпусом, правильна відмова", "За наявності попиту додати окремий перевірений розділ."
        return "поза корпусом, відповідь без доказу", "Перевіряти, чи цитата прямо відповідає на запит."
    if pd.isna(row["ранг доказу"]):
        return "промах ретриву", "Додати синоніми до заголовка й перевірити більший k."
    if row["стан"] in {"відхилено цитату або джерело", "помилка формату"}:
        return "помилка формату або походження цитати", "Використати обмежене JSON-декодування та один повтор."
    if not row["збіг ознак"]:
        return "помилка генерації за наявного доказу", "Перевіряти числові параметри і повноту відповіді."
    return "цитата не містить еталонного доказу", "Просити повне речення з потрібним фактом."

failures = results[
    ~results["збіг ознак"] | results["джерело"].isna() | results["цитування"].eq(False)
].copy()
failures[["тип", "зміна"]] = failures.apply(diagnose, axis=1, result_type="expand")
display(failures[["№", "питання", "відповідь", "ранг доказу", "тип", "зміна"]])

,№,питання,відповідь,ранг доказу,тип,зміна
4,5,"Як отримати результат, придатний для автоматич...",У наданих джерелах немає відповіді.,NaN,промах ретриву,Додати синоніми до заголовка й перевірити біль...
9,10,Скільки клієнт чекає завершення запиту до Ollama?,У наданих джерелах немає відповіді.,2.0,помилка генерації за наявного доказу,Перевіряти числові параметри і повноту відповіді.
12,13,"Які слова вилучає CLI, коли перетворює твердже...","CLI вилучає всі слова, крім двох останніх, які...",1.0,помилка генерації за наявного доказу,Перевіряти числові параметри і повноту відповіді.
17,18,Який повний результат у тесті якості здобули м...,"Модель 1.5B отримала 7 балів з 10, модель 3B —...",1.0,помилка генерації за наявного доказу,Перевіряти числові параметри і повноту відповіді.
20,21,Який тариф за тисячу токенів у хмарній версії ...,У наданих джерелах немає відповіді.,NaN,"поза корпусом, правильна відмова",За наявності попиту додати окремий перевірений...
21,22,Як експортувати картки у форматі Anki APKG?,У наданих джерелах немає відповіді.,NaN,"поза корпусом, правильна відмова",За наявності попиту додати окремий перевірений...
22,23,Які облікові дані потрібні для вебкабінету?,У наданих джерелах немає відповіді.,NaN,"поза корпусом, правильна відмова",За наявності попиту додати окремий перевірений...
23,24,Чи можна подати скан PDF і який OCR використов...,У наданих джерелах немає відповіді.,NaN,"поза корпусом, правильна відмова",За наявності попиту додати окремий перевірений...
24,25,Яка точність генерації на довгих конспектах ні...,У наданих джерелах немає відповіді.,NaN,"поза корпусом, правильна відмова",За наявності попиту додати окремий перевірений...
25,26,"Яку команду виконати, щоб опублікувати картки ...","Щоб опублікувати картки для одногрупників, пот...",NaN,"поза корпусом, відповідь без доказу","Перевіряти, чи цитата прямо відповідає на запит."


## 1.7. Демонстрація

Поле приймає довільне запитання. Кнопка показує відповідь, дослівну цитату й посилання.

In [16]:
import ipywidgets as widgets

field = widgets.Text(
    value="Скільки триває таймаут запиту до Ollama?",
    description="Питання", layout=widgets.Layout(width="95%"),
)
button = widgets.Button(description="Запитати")
output = widgets.Output()

def on_question(_):
    question = field.value.strip()
    with output:
        output.clear_output()
        if not question:
            print("Введіть питання.")
            return
        button.disabled = True
        try:
            answer, source, quote, _, _, _ = respond(question)
            print(answer)
            if source:
                print(f"Цитата: {quote}")
                display(Markdown(f"[{source.title}]({source.url})"))
        except requests.RequestException as error:
            print(f"Помилка з'єднання з API: {error}")
        finally:
            button.disabled = False

button.on_click(on_question)
display(widgets.VBox([field, button, output]))

## 1.8. Мій звіт

### Корпус і бенчмарк

Шість Markdown-конспектів з лабораторної 4 охоплюють параметри CLI, обробку помилок і перевірку якості. Обсяг корпусу — 4540 байтів у 18 тематичних розділах. RAG потрібен, щоб одразу показати перевірний уривок і посилання на зафіксовану версію джерела. Бенчмарк зафіксовано перед налаштуванням індексу. У ньому 20 питань з еталоном і шість поза корпусом.

### Індексація й ретрив

Multilingual E5 Small дає 384-вимірні нормалізовані вектори. Є 18 фрагментів. Максимум за правилом поділу — 64 слова з перекриттям 12, а фактичний максимум — 27 слів, тому в цьому корпусі кожен розділ утворив один фрагмент без внутрішнього перекриття. Точний пошук працює через матрицю NumPy.

| Пошук | hit@1 | hit@3 | hit@5 | MRR@5 |
| --- | ---: | ---: | ---: | ---: |
| Векторний | 0,75 | 0,85 | 0,85 | 0,792 |
| Векторний + BM25 через RRF | 0,85 | 0,90 | 0,95 | 0,888 |

Для генерації використано гібридний пошук, який повернув доказ для 19 із 20 питань. Конфігурацію вибрано на цьому самому наборі. Результати характеризують цей корпус і бенчмарк.

### Генерація

| Конфігурація | Збіги за автоматичним критерієм | Цитати з еталонним доказом | Правильні відмови |
| --- | ---: | ---: | ---: |
| Qwen2.5 1.5B, початковий промпт | 10/20 | 9/20 | 2/6 |
| Qwen3 4B, уточнений промпт і перевірка чисел у цитаті | 16/20 | 18/20 | 5/6 |

Ручна звірка збільшує оцінку фінальної конфігурації до 17/20. У №18 модель правильно навела обидва результати, але шаблон автоматичної оцінки пропустив формулювання «7 балів з 10» і «2 балів з 10». Із тих самих п'яти фрагментів модель генерує відповідь і дослівну цитату. Перевірка точної цитати та її чисел обмежує вигадані посилання, але не перевіряє, чи відповідь логічно випливає з цитати. Генерація 26 відповідей у фінальному запуску на T4 зайняла 140,4 с.

### Розбір відмов і помилок

| № | Спостереження | Причина й можлива зміна |
| --- | --- | --- |
| 5 | Відмова на питання про машинозчитуваний результат. | Фрагмента з опцією --json не було в топ-5. Додати синоніми «машинозчитуваний», «автоматична обробка» до заголовка розділу. |
| 10 | Відмова про час очікування. | Правильні 60 секунд були у фрагменті на другій позиції. Допоможе переранжування доказів за відповідністю питанню про тривалість. |
| 13 | Відповідь суперечливо описує, які слова вилучає CLI. | Доказ був на першій позиції. Для точного правила перетворення краще повертати речення з джерела або структуроване значення «два останні слова». |
| 18 | Автоматична метрика відхилила коректне перефразування двох оцінок. | Це помилка оцінки. Порівняння нормалізованих числових пар або ручна звірка врахує текст між чисельником і знаменником. |
| 22 | Коректна відмова щодо Anki APKG. | У джерелах описано JSON, а формат APKG не задокументовано. За потреби додати окремий розділ про експорт, якщо його буде реалізовано. |
| 24 | Коректна відмова щодо PDF і OCR. | Корпус описує текстовий файл та stdin. Окремий розділ із переліком підтриманих форматів зробив би межу явною. |
| 26 | Команду створення карток подано як спосіб публікації. | Цитата справжня, але підтверджує запуск CLI, а не публікацію. Потрібна перевірка, що цитата прямо обґрунтовує дію з питання. |

Гібридний пошук дав більше доказів у верхніх позиціях. Основні помилки залишилися на переході від уривка до відповіді, зокрема коли уривок стосується подібної дії. Окремі метрики збігу відповіді, цитування та відмов показують ці проблеми краще, ніж одна загальна точність.

In [17]:
display(Markdown(
    f"- **Індексація.** 6 документів, {bytes_count} байтів, {len(chunks)} фрагментів. "
    f"Модель multilingual-e5-small, {vectors.shape[1]} ознак.\n"
    f"- **Ретрив.** Dense hit@1 {retrieval_metrics.loc['dense', 'hit@1']:.2f}, "
    f"hit@5 {retrieval_metrics.loc['dense', 'hit@5']:.2f}, MRR@5 {retrieval_metrics.loc['dense', 'MRR@5']:.3f}. "
    f"Hybrid hit@1 {retrieval_metrics.loc['hybrid', 'hit@1']:.2f}, "
    f"hit@5 {retrieval_metrics.loc['hybrid', 'hit@5']:.2f}, MRR@5 {retrieval_metrics.loc['hybrid', 'MRR@5']:.3f}.\n"
    f"- **Генерація.** {model_name}, пошук {METHOD}. "
    f"Ознаки правильних відповідей {int(measured['збіг ознак'].sum())}/{len(measured)}, "
    f"коректні цитати {int(measured['цитування'].sum())}/{len(measured)}, "
    f"правильні відмови {int(outside['збіг ознак'].sum())}/{len(outside)}.\n"
    f"- **Відмови.** {len(failures)} випадків для діагностики. "
    f"{int(failures['тип'].str.startswith('поза корпусом').sum())} стосуються меж корпусу.\n"
    "- **Висновок.** Цитата дає можливість перевірити числові параметри, команди й джерела. "
    "Ознаки правильності слід доповнити ручною оцінкою та перевіркою на нових питаннях."
))

- **Індексація.** 6 документів, 4540 байтів, 18 фрагментів. Модель multilingual-e5-small, 384 ознак.
- **Ретрив.** Dense hit@1 0.75, hit@5 0.85, MRR@5 0.792. Hybrid hit@1 0.85, hit@5 0.95, MRR@5 0.887.
- **Генерація.** Qwen/Qwen3-4B-Instruct-2507, пошук hybrid. Ознаки правильних відповідей 16/20, коректні цитати 18/20, правильні відмови 5/6.
- **Відмови.** 10 випадків для діагностики. 6 стосуються меж корпусу.
- **Висновок.** Цитата дає можливість перевірити числові параметри, команди й джерела. Ознаки правильності слід доповнити ручною оцінкою та перевіркою на нових питаннях.